In [ ]:
# Core imports
import os
import json
import math
import time
import textwrap
import warnings
from pathlib import Path

import torch
import ipywidgets as widgets
import matplotlib.pyplot as plt

from IPython.display import display, clear_output, Markdown

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    set_seed,
)

warnings.filterwarnings("ignore")
set_seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


In [ ]:
# Curated model catalog
MODEL_CATALOG = {
    "Qwen2.5-3B-Instruct": {
        "hf_id": "Qwen/Qwen2.5-3B-Instruct",
        "best_for": "Strong general-purpose instruct model; good default for chat, explanation, multilingual prompts, and structured answers.",
        "size_note": "About 3B parameters; practical for many local experiments.",
        "license_note": "Open-weight style usage on the Hub; check the model card for current terms.",
    },
    "Mistral-7B-Instruct-v0.3": {
        "hf_id": "mistralai/Mistral-7B-Instruct-v0.3",
        "best_for": "General instruct chat, strong assistant behavior, and useful experiments with function-calling style prompting.",
        "size_note": "7B model; likely heavier on RAM / VRAM.",
        "license_note": "Apache-2.0 on the model card.",
    },
    "Phi-3.5-mini-instruct": {
        "hf_id": "microsoft/Phi-3.5-mini-instruct",
        "best_for": "Compact local experimentation, concise instruction following, and lighter setups.",
        "size_note": "Compact model family with long-context support according to the model card.",
        "license_note": "MIT according to the model card.",
    },
    "Llama-3.2-3B-Instruct": {
        "hf_id": "meta-llama/Llama-3.2-3B-Instruct",
        "best_for": "General assistant tasks, summarization, rewriting, and multilingual dialogue.",
        "size_note": "3B model, but gated access may apply.",
        "license_note": "Check access conditions on the model page before use.",
    },
    "Gemma-2-2B-it": {
        "hf_id": "google/gemma-2-2b-it",
        "best_for": "Lightweight local instruction-following experiments.",
        "size_note": "Smaller model; good when memory is tight.",
        "license_note": "Requires accepting the Gemma license on the Hub.",
    },
    "Qwen2.5-Coder-3B-Instruct": {
        "hf_id": "Qwen/Qwen2.5-Coder-3B-Instruct",
        "best_for": "Code generation, code explanation, and code-fixing experiments.",
        "size_note": "3B code-specialized instruct model.",
        "license_note": "Check the model card for current license terms.",
    },
    "DeepSeek-R1-Distill-Qwen-7B": {
        "hf_id": "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
        "best_for": "Reasoning-heavy experiments, math-style prompting, and reflective answers.",
        "size_note": "7B reasoning-oriented distilled model; heavier than the compact options.",
        "license_note": "MIT on the model card, with lineage notes documented there.",
    },
}


In [ ]:
# Display the model catalog in a clean formatted way
for model_name, info in MODEL_CATALOG.items():
    print("=" * 100)
    print(model_name)
    print("  HF id:        ", info["hf_id"])
    print("  Best for:     ", info["best_for"])
    print("  Size note:    ", info["size_note"])
    print("  License note: ", info["license_note"])


In [ ]:
# Dropdown widget for selecting a model from the catalog
model_dropdown = widgets.Dropdown(
    options=list(MODEL_CATALOG.keys()),
    value="Qwen2.5-3B-Instruct",
    description="Model:",
    style={'description_width': '80px'},
    layout=widgets.Layout(width='450px')
)

display(model_dropdown)


In [6]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import time

load_button_fake = widgets.Button(
    description="Load Model",
    button_style="success",
    layout=widgets.Layout(width='200px')
)

load_status_fake = widgets.Output()

loaded_model = "FAKE_MODEL"
loaded_tokenizer = "FAKE_TOKENIZER"

def fake_load(b):
    load_status_fake.clear_output()
    with load_status_fake:
        print("Loading model... please wait.")
    time.sleep(2)
    with load_status_fake:
        load_status_fake.clear_output()
        print("Model loaded successfully! (demo mode)")

load_button_fake.on_click(fake_load)

display(load_button_fake, load_status_fake)


Button(button_style='success', description='Load Model', layout=Layout(width='200px'), style=ButtonStyle())

Output()

In [7]:
from IPython.display import Markdown, display

chat_template_preview = """
### Chat Template Preview

This is how the prompt will be formatted before sending it to the model:

<|user|>
Hello, how are you?

<|assistant|>
"""

display(Markdown(chat_template_preview))



### Chat Template Preview

This is how the prompt will be formatted before sending it to the model:

<|user|>
Hello, how are you?

<|assistant|>


In [8]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Поле ввода сообщения
user_input = widgets.Textarea(
    placeholder="Type your message here...",
    layout=widgets.Layout(width='600px', height='100px')
)

# Кнопка отправки
send_button = widgets.Button(
    description="Send",
    button_style="primary",
    layout=widgets.Layout(width='150px')
)

# Область вывода чата
chat_output = widgets.Output()

def on_send_clicked(b):
    message = user_input.value.strip()
    if not message:
        return

    with chat_output:
        print(f"User: {message}")
        print("Assistant: (demo response) This is a placeholder reply.\n")

    user_input.value = ""

send_button.on_click(on_send_clicked)

display(user_input, send_button, chat_output)


Textarea(value='', layout=Layout(height='100px', width='600px'), placeholder='Type your message here...')

Button(button_style='primary', description='Send', layout=Layout(width='150px'), style=ButtonStyle())

Output()